# Tutorial 2: reinforcement learning with GRPO

In notebook 1 the model learnt by copying worked solutions. That teaches a
format well, but the model can only be as good as its examples, and it is
never told whether its own answers are right. *Reinforcement learning* (RL)
works the other way round: the model writes its own answers, a program checks
them, and the model is nudged towards the answers that scored well. It needs
no worked solutions, only questions and a way to check an answer.

**GRPO** (Group Relative Policy Optimisation) is the RL method DeepSeek used
to train DeepSeek-R1. Each training step:

1. **samples**: for each question, the model writes a *group* of answers
   (eight here);
2. **scores**: *reward functions* give each answer a number;
3. **compares**: each answer's *advantage* is how much better it scored than
   the rest of its group;
4. **updates**: answers with a positive advantage are made more likely and
   those with a negative one less likely, while a penalty stops the model
   drifting too far from where it started.

You will look at the prompts and the rewards, work out advantages by hand,
train for 200 steps, and compare the base, SFT and GRPO models.

**Before you start**, run `01_sft.ipynb`: this notebook starts from the model
it trained and uses the data it wrote. One TPU v6e chip is enough, and nothing
else may be using the TPU.

## 1. Set up

In [ ]:
import os
import sys
from pathlib import Path

# Jupyter starts in examples/; every command below runs from the repository
# root, as the README's do.
if not Path("pyproject.toml").is_file():
    os.chdir("..")
assert Path("src/open_r1_tpu").is_dir(), f"Not an open-r1-tpu checkout: {Path.cwd()}"
sys.path.insert(0, str(Path("examples").resolve()))
print("Working in", Path.cwd())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from notebook_utils import count_tpu_chips, plot_scalars, read_scalars, run

PYTHON = sys.executable  # the Python that runs this notebook, and the commands

SFT_MODEL = Path("artifacts/Qwen2.5-0.5B-GSM8K-SFT/merged")
for needed in (
    SFT_MODEL / "model.safetensors",
    Path("data/gsm8k-think/train.parquet"),
    Path("data/gsm8k-think/test.parquet"),
):
    assert needed.exists(), f"{needed} is missing: run 01_sft.ipynb first"

In [ ]:
CHIPS = count_tpu_chips()
MESH = f"model.mesh.shape=[{CHIPS},1]"
print(f"{CHIPS} chip(s); recipe override: {MESH}")

## 2. Prompts and right answers

For each question GRPO needs only the text of the question and the right
answer. The prompt is rendered exactly as in SFT, with the same system prompt
and chat template, and ends where the model starts writing:

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

from open_r1_tpu.core.config import read_prompt_file
from open_r1_tpu.grpo.data import render_prompt

train = load_dataset(
    "parquet", data_files="data/gsm8k-think/train.parquet", split="train"
)
tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL)
SYSTEM_PROMPT = read_prompt_file("examples/recipes/system_prompt.txt")

row = train[0]
print(render_prompt(row["question"], tokenizer, system_prompt=SYSTEM_PROMPT))
print("Right answer:", row["answer"])

## 3. Rewards

A reward function takes the model's answers and returns a score for each.
This tutorial adds up two of the functions in
`src/open_r1_tpu/grpo/rewards.py`:

- `format_reward`: 3 for a closed `<think> ... </think>` block followed by a
  `\boxed{}` answer. Otherwise half a point up or down for each part present
  or missing, so an answer close to the right shape still beats one far from
  it.
- `correctness_reward`: 3 when the boxed answer is right, 0.5 when it is
  within 10%, -0.5 when it is wrong, and 0 when there is no boxed answer.

Try them on some hand-written answers to the question above:

In [ ]:
from open_r1_tpu.grpo.rewards import correctness_reward, format_reward

answers = {
    "right, formatted": "<think>\n48/2 = 24\n48+24 = 72\n</think>\nSo \\boxed{72}.",
    "wrong, formatted": "<think>\n48*2 = 96\n48+96 = 144\n</think>\nSo \\boxed{144}.",
    "close, formatted": "<think>\n48/2 = 24\n48+24 = 70\n</think>\nSo \\boxed{70}.",
    "right, unformatted": "She sold 72 clips altogether.",
    "cut off": "<think>\n48/2 = 24\n48+24 =",
}
completions = list(answers.values())
prompts = [""] * len(completions)
gold = [row["answer"]] * len(completions)
pd.DataFrame(
    {
        "format_reward": format_reward(prompts=prompts, completions=completions),
        "correctness_reward": correctness_reward(
            prompts=prompts, completions=completions, answer=gold
        ),
    },
    index=list(answers),
).assign(total=lambda table: table.sum(axis=1))

The reward is the only thing that tells the model what you want, and the
model will exploit any gap in it, so test a reward on examples like these
before training with it. Here, for instance, a right answer without the
format scores below a wrong answer with it.

## 4. Advantages: comparing answers within a group

Say the model writes eight answers to one question, three of them right.
GRPO scores each answer against its own group: the *advantage* is the
answer's reward minus the group's mean, divided by the group's standard
deviation.

In [ ]:
rewards = np.array([6.0, 2.5, 2.5, 6.0, 2.5, 3.5, 2.5, 6.0])


def advantages(rewards):
    """GRPO's advantage, as Tunix computes it for one group."""
    return (rewards - rewards.mean()) / (rewards.std(ddof=1) + 1e-6)


pd.DataFrame({"reward": rewards, "advantage": advantages(rewards).round(2)})

The right answers get a positive advantage and become more likely; the wrong
ones get a negative advantage and become less likely. The group is its own
baseline, so GRPO needs no second network to estimate how good an answer
should have been, which is what makes it simpler than older methods such as
PPO.

Now a group in which every answer is right:

In [ ]:
advantages(np.full(8, 6.0))

Every advantage is zero, so this group teaches nothing; the same holds when
every answer is wrong. Questions that are far too easy or far too hard waste
the step, and answers must vary, which is why they are sampled at a
temperature rather than chosen greedily. Training logs the fraction of groups
with mixed rewards as `signal/groups_mixed_frac`.

## 5. The policy and the reference

GRPO keeps two copies of the SFT model on the TPU:

- the **policy**, the model being trained. Only a *LoRA adapter* trains: a
  pair of thin matrices beside each weight matrix of each layer (32 is their
  width, the *rank*), 18 million numbers against the model's 494 million.
  The original weights stay frozen.
- the **reference**, a frozen copy. A *KL penalty* (`grpo.beta`) measures how
  far the policy's token probabilities have moved from the reference's, and
  counts against the reward, so the model cannot forget everything else in
  pursuit of it.

A *clip* (`grpo.epsilon`) also limits how far one update can change any
answer's probability.

## 6. The recipe

In [ ]:
GRPO_RECIPE = "examples/recipes/grpo-gsm8k.yaml"
print(Path(GRPO_RECIPE).read_text())

Compared with the SFT recipe there are two new sections:

| Section | What it sets |
| --- | --- |
| `grpo` | the reward functions, the group size, the KL weight and the clip |
| `rollout` | how answers are sampled: prompt and answer lengths, temperature, where to stop |

`training` also gains *micro-batch* sizes: how many questions go through the
chip at once. A step's 8 questions x 8 answers are processed in pieces and
their gradients added up, which keeps the memory use in check.

Two settings are not choices. On the pinned version of Tunix, Qwen2.5 models
need `dtype: float32` and `use_flash_attention: false` for GRPO; with
`bfloat16` or the splash attention kernel the sampled answers come out as
gibberish.

In [ ]:
from open_r1_tpu.core.config import load_config
from open_r1_tpu.grpo.config import validate_grpo_config

config = load_config(GRPO_RECIPE, [MESH], validator=validate_grpo_config)
questions = config["dataset"]["batch_size"]
group = config["grpo"]["num_generations"]
print(
    f"{questions} questions x {group} answers = {questions * group} answers "
    f"per step, for {config['training']['max_steps']} steps"
)

## 7. Train

The run loads the SFT model twice (the policy, with its new LoRA adapter, and
the reference), then evaluates: the model answers 32 held-out questions and
their rewards are logged. Each step then samples 64 answers, scores them,
computes advantages and updates the adapter. Every 50 steps it evaluates
again. At the end the adapter is merged into the SFT weights and exported to
`artifacts/Qwen2.5-0.5B-GSM8K-GRPO/merged`.

Sampling answers is most of the work, so a GRPO step takes much longer than
an SFT step. As before, interrupting the cell stops the run, running it again
resumes from the newest checkpoint, and deleting
`artifacts/Qwen2.5-0.5B-GSM8K-GRPO` starts again.

In [ ]:
run(PYTHON, "-m", "open_r1_tpu.grpo.run", "--config", GRPO_RECIPE, MESH)

## 8. Watch the rewards

In [ ]:
GRPO_DIR = Path("artifacts/Qwen2.5-0.5B-GSM8K-GRPO")
scalars = read_scalars(GRPO_DIR / "logs")

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
plot_scalars(
    scalars,
    ["rewards/train/correctness_reward", "rewards/train/format_reward"],
    title="Training answers: mean reward",
    ax=axes[0, 0],
)
plot_scalars(
    scalars,
    ["rewards/eval/correctness_reward", "rewards/eval/format_reward"],
    title="Held-out answers: mean reward",
    ax=axes[0, 1],
)
plot_scalars(
    scalars,
    ["completions/train/mean_length"],
    title="Answer length (tokens)",
    ax=axes[1, 0],
)
plot_scalars(
    scalars,
    ["signal/train/groups_mixed_frac"],
    title="Fraction of groups that teach something",
    ax=axes[1, 1],
)
fig.tight_layout()

- **Rewards on training answers** are noisy, because every step brings eight
  new questions; look at the trend. A rising `correctness_reward` means the
  model is getting more answers right. `format_reward` should start close to
  its maximum of 3, since SFT already taught the format.
- **Rewards on held-out answers** come from the same 32 questions at every
  evaluation, so they compare like with like.
- **Answer length** shows whether the model is learning to write more or less
  working.
- **Groups that teach something** is the fraction with any spread in reward.
  If it fell towards zero, most steps would teach nothing.

The table of every logged metric is in `scalars`; `scalars.tag.unique()`
lists them.

## 9. Read what the model wrote

The numbers say *whether* the model changed; the answers say *how*. Every
evaluation answer was saved, with its rewards, to `eval_rollouts.jsonl`.

In [ ]:
rollouts = pd.read_json(GRPO_DIR / "eval_rollouts.jsonl", lines=True)
rollouts["right"] = rollouts.rewards.map(lambda r: r["correctness_reward"] == 3.0)
rollouts.groupby("step").agg(mean_reward=("reward", "mean"), right=("right", "mean"))

The same question at the first and the last evaluation:

In [ ]:
question = rollouts.question.iloc[0]
print(question, "\nRight answer:", rollouts.answer.iloc[0])
for step in (rollouts.step.min(), rollouts.step.max()):
    answer = rollouts[(rollouts.step == step) & (rollouts.question == question)].iloc[0]
    print(f"\n--- step {step}, reward {answer.reward} ---")
    print(answer.completion)

## 10. Base, SFT and GRPO on the test set

Finally, answer the 1,319 GSM8K test questions, which training never saw,
with the GRPO model, and compare it with the two models from notebook 1.

In [ ]:
ANSWERS = Path("artifacts/gsm8k-answers")


def answer_test_questions(model_path, name):
    """Ask a model every GSM8K test question and load its scored replies."""
    output = ANSWERS / f"{name}.jsonl"
    run(
        PYTHON,
        "examples/answer_questions.py",
        model_path,
        "--model-name",
        "qwen2.5-0.5b",
        "--questions",
        "data/gsm8k-think/test.parquet",
        "--system-prompt-file",
        "examples/recipes/system_prompt.txt",
        "--output",
        output,
    )
    return pd.read_json(output, lines=True)

In [ ]:
GRPO_MODEL = GRPO_DIR / "merged"
results = {"GRPO": answer_test_questions(GRPO_MODEL, "grpo")}
for name, model in (("SFT", SFT_MODEL), ("base", Path("models/Qwen2.5-0.5B"))):
    saved = ANSWERS / f"{name.lower()}.jsonl"
    if saved.exists():  # answered in notebook 1
        results[name] = pd.read_json(saved, lines=True)
    elif model.exists():
        results[name] = answer_test_questions(model, name.lower())

In [ ]:
pd.DataFrame(
    {
        name: [answers.correct.mean(), answers.formatted.mean()]
        for name, answers in reversed(results.items())
    },
    index=["correct", "formatted"],
).style.format("{:.1%}")

Is a difference of a point or two real, or luck? Since both models answered
the same questions, count the questions whose result changed:

In [ ]:
pd.crosstab(
    results["SFT"].correct,
    results["GRPO"].correct,
    rownames=["right after SFT"],
    colnames=["right after GRPO"],
)

The off-diagonal cells are the questions that changed. If many more went from
wrong to right than the other way, the improvement is real; if the two are
close, it may be noise. Each question was answered once and greedily, so for
numbers you would report, use the repository's evaluation, which samples
several times with different seeds (`docs/evaluation.md`).

## Next steps

Each of these is an override added to the training command, for example
`run(PYTHON, "-m", "open_r1_tpu.grpo.run", "--config", GRPO_RECIPE, MESH,
"training.max_steps=400")`.

- **Train for longer** with `training.max_steps=400`: the run resumes from its
  last checkpoint.
- **Change the reward** with `grpo.reward_functions=[answer_correctness_reward]`,
  which scores only whether the final number is right, with no format
  requirement. To write your own, add a function to
  `src/open_r1_tpu/grpo/rewards.py` and register it in `REWARD_FNS_BY_NAME`.
- **Change the group** with `grpo.num_generations=4`, which halves the
  sampling per question; watch what happens to `signal/groups_mixed_frac`.

Delete `artifacts/Qwen2.5-0.5B-GSM8K-GRPO` before changing the reward or the
group, so the run starts again from the SFT model.
- **Go bigger**: `recipes/Qwen2.5-1.5B/grpo/simplerl-zoo.yaml` reproduces a
  published GRPO result on four chips; the README's GRPO section describes it.